# Model Comparison

Head-to-head benchmark of every model on the same 70/30 split with random_state=67.

Results are written to `reports/model_comparison.csv`.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from src.preprocessing import build_features
from src.evaluate import fit_score

X, y, _ = build_features('../data/video_game_reviews.csv')
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=67)

In [ ]:
results = []
results.append(fit_score(DummyRegressor(strategy='mean'), Xtr, Xte, ytr, yte, 'Baseline (mean)'))
results.append(fit_score(DecisionTreeRegressor(max_depth=20, min_samples_leaf=50, random_state=67), Xtr, Xte, ytr, yte, 'Decision Tree (tuned)'))
results.append(fit_score(RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1), Xtr, Xte, ytr, yte, 'Random Forest'))
results.append(fit_score(HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_depth=8, random_state=0), Xtr, Xte, ytr, yte, 'HistGradientBoosting'))
results.append(fit_score(GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=0), Xtr, Xte, ytr, yte, 'GradientBoosting (sklearn)'))

scaler = StandardScaler()
Xtr_s, Xte_s = scaler.fit_transform(Xtr), scaler.transform(Xte)
results.append(fit_score(MLPRegressor(hidden_layer_sizes=(16,), activation='relu', solver='adam', max_iter=300, random_state=1), Xtr_s, Xte_s, ytr, yte, 'Neural Network (MLP-16)'))

df = pd.DataFrame([r.as_row() for r in results]).sort_values('r2', ascending=False).reset_index(drop=True)
df

In [ ]:
df.to_csv('../reports/model_comparison.csv', index=False)